In [3]:
import json

BASE = "../data/raw"
with open(f"{BASE}/metadata/questions.json", encoding="utf-8") as f:
    questions = json.load(f)

print(len(questions))

7652


In [ ]:
#문항 하나만 구조 확인
qid = next(iter(questions))
print(qid)
print(questions[qid].keys())
print(questions[qid]["content"][:200])
print(questions[qid]["analysis"][:200])

0
dict_keys(['content', 'kc_routes', 'answer', 'analysis', 'type', 'options'])
学校有舞蹈，唱歌、围棋、绘画四种兴趣班． 小宇、小明、小丽三个小朋友准备报名，每人只能报一个班而且各不相同，一共有种不同的报名方法．
小宇先选，有$$4$$种，接下来小明选，有$$3$$种，最后小丽选，有$$2$$种，所以一共有$$4\times3\times2=24$$（种）．


In [5]:
# 해설이 있는 문항 수
with_analysis = [k for k, v in questions.items() if v.get("analysis")]
print(len(with_analysis), "/", len(questions))

7652 / 7652


In [6]:
#이미지를 가리키는 표시 찾기
import re
pat = re.compile(r"(\.jpg|\.png|<img|\[图|image)", re.I)
hits = [k for k, v in questions.items() if pat.search(v.get("content", ""))]
print(len(hits))
print(questions[hits[0]]["content"][:300] if hits else "none")

2189
用$$5$$种不同的颜色给下面的图形染色，要求相邻的区域（有公共边的两区域称为相邻）染成不同的颜色．如果颜色能重复使用，则共有种不同的染色方法．
 question_2-image_0


키 구성은 content, kc_routes, answer, analysis, type, options

해설은 7,652개 전부 있습니다. 해설이 빈 문항 걱정은 안 해도 됩니다.

본문은 중국어이고, 수식은 $$...$$ 형태의 LaTeX입니다. 해설도 풀이 과정이 수식으로 적혀 있습니다.

이미지 표시는 question_2-image_0 같은 형태로 본문에 들어 있습니다. 앞서 쓴 정규식에는 image라는 글자가 포함돼 있어서 이 표시를 잡았습니다.

정규식을 question_\d+-image_\d+ 같은 표시만 잡도록 좁힙니다. 표시 형식이 항상 이렇게 생겼는지 몇 개 더 확인하세요.

해설에도 이미지 표시가 있는지 같이 봅니다. 본문이 텍스트여도 해설이 그림을 참조하면 RAG 답변이 어색해집니다.

본문과 해설 둘 다 이미지 표시가 없는 문항만 골라 data/processed/에 Parquet으로 저장합니다.

In [7]:
import re
import pandas as pd

img = re.compile(r"question_\d+-image_\d+|analysis_\d+-image_\d+")

rows = []
for qid, v in questions.items():
    c, a = v.get("content", ""), v.get("analysis", "")
    rows.append({
        "qid": int(qid), "content": c, "analysis": a, "answer": v.get("answer"),
        "type": v.get("type"), "kc_routes": v.get("kc_routes"),
        "has_img_content": bool(img.search(c)),
        "has_img_analysis": bool(img.search(a)),
    })

df = pd.DataFrame(rows)
print(df[["has_img_content", "has_img_analysis"]].value_counts())

has_img_content  has_img_analysis
False            False               4941
True             False               1574
                 True                 615
False            True                 522
Name: count, dtype: int64


| 구분              | 문항 수  | 비율      |
| --------------- | ----- | ------- |
| 본문·해설 모두 이미지 없음 | 4,941 | 약 64.6% |
| 본문에만 이미지        | 1,574 | 약 20.6% |
| 본문·해설 모두 이미지    | 615   | 약 8.0%  |
| 해설에만 이미지        | 522   | 약 6.8%  |

In [8]:
df_all = df.copy()
df_rag = df[~df["has_img_content"] & ~df["has_img_analysis"]].copy()

df_all.to_parquet("../data/processed/questions_all.parquet", index=False)
df_rag.to_parquet("../data/processed/rag_questions.parquet", index=False)
print(len(df_all), len(df_rag))

7652 4941


In [ ]:
#풀이 로그 읽고 구조 보기
import pandas as pd

seq = pd.read_csv(f"{BASE}/question_level/train_valid_sequences_quelevel.csv")
print(seq.shape)
print(seq.columns.tolist())
print(seq["questions"].iloc[0][:200])

(30965, 7)
['fold', 'uid', 'questions', 'concepts', 'responses', 'timestamps', 'selectmasks']
3751,3752,3753,3754,1990,3739,3740,3742,3756,3757,3600,3758,3760,3761,3925,3928,1916,1914,3764,3763,3767,3766,3768,3931,3933,3772,5749,4664,5750,5751,5752,5754,5755,6,3934,3935,3936,3937,3941,3943,394


In [10]:
#문항 번호를 숫자로 풀기
def parse_ids(s):
    return [int(x) for x in str(s).split(",") if x.strip() not in ("", "-1")]

all_ids = set()
for s in seq["questions"]:
    all_ids.update(parse_ids(s))

print(len(all_ids))

7618


In [11]:
#문항 정보와 대조
known = set(int(k) for k in questions.keys())
missing = all_ids - known
unused = known - all_ids

print("로그에 있는데 문항 정보에 없는 번호:", len(missing))
print("문항 정보에 있는데 로그에 안 나오는 번호:", len(unused))
print(sorted(missing)[:10])

로그에 있는데 문항 정보에 없는 번호: 0
문항 정보에 있는데 로그에 안 나오는 번호: 34
[]
